# Validasi 4-Fold CV Penuh — 3 Kandidat Teratas

Notebook ini menjawab pertanyaan yang diajukan di akhir
[Laporan XAI](https://github.com/Ardiyanto24/coffee-bean-quality-detection/blob/main/reports/CBQD%20-%20XAI%20Report.html):
**apakah peringkat macro-F1 `09_noise_robust` > `05_convnext_tiny` > `08_multitask` di
`CBQD - Training.ipynb` cuma kebetulan dari SATU pembagian fit/val, atau benar-benar stabil?**

`CBQD - Training.ipynb` sendiri sudah eksplisit menyatakan keterbatasannya: *"1 split
screening, bukan full 4-fold CV (kelayakan waktu untuk 10 model sekaligus)"*. Menguji 3
model (bukan 10) membuat full 4-fold CV genuinely feasible.

**Metodologi:**
- `dataset_preprocessed/train/` (929 gambar) sudah punya kolom `cv_fold` (0-3, stratified
  per kelas ~232 gambar/fold — dihasilkan `CBQD - Preprocessing.ipynb`, BUKAN dibuat baru
  di sini). Untuk tiap fold *k* ∈ {0,1,2,3}: `cv_fold==k` jadi **val** (~232), `cv_fold≠k`
  jadi **fit** (~697) — persis definisi 4-fold CV standar, dirotasi penuh 4×.
- `dataset_preprocessed/test/` (231 gambar) TETAP jadi held-out test yang SAMA di semua
  4 fold — tidak pernah ikut fit/val di fold manapun. Ini yang membuat perbandingan
  test-macro-F1 antar-fold bermakna: yang berubah cuma data TRAINING, evaluasinya tetap
  di target yang identik.
- Ketiga model dilatih PERSIS dengan prosedur yang sama seperti `CBQD - Training.ipynb`
  (arsitektur, augmentasi, 2-fase fine-tune, hyperparameter, early stopping) — satu-satunya
  yang berubah adalah komposisi fit/val per fold. Untuk `09_noise_robust`, skor
  mistakenness (RandomForest OOF cluster-aware) **dihitung ulang per fold** karena proxy
  ini memang didefinisikan relatif terhadap fit pool fold itu sendiri.
- Checkpoint 12 model hasil (3 model × 4 fold) TIDAK disimpan ke DVC -- nilainya ada di
  metrik agregat, bukan model individualnya (beda dari `CBQD - Training.ipynb` yang
  checkpoint-nya dipakai ulang `CBQD - XAI.ipynb`).

**PENTING — DRY_RUN:** sama seperti notebook training/XAI sebelumnya. `DRY_RUN=True` ->
cuma 2 fold + epoch kecil, untuk memastikan seluruh loop 3-model x N-fold jalan tanpa error
dari awal sampai akhir. Baru ubah ke `DRY_RUN=False` untuk 4 fold penuh + 50 epoch/model
(total 12 training run penuh -- notebook ini akan berjalan lama, dan itu tidak masalah).

## Section 1 — Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Clone repo (tanpa menyentuh torch/torchvision -- sklearn & opencv sudah bawaan image Kaggle)

import os
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

import json

_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")

In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset + manifest dari R2 (dvc pull)

!pip install -q "dvc[s3]"
!dvc pull -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())

## Section 2 — Konfigurasi Eksperimen

Hyperparameter PERSIS sama dengan `CBQD - Training.ipynb` -- supaya perbandingan
antar-fold murni mengukur efek pembagian data, bukan efek pengaturan yang berbeda.

In [ ]:
# Sub-Step 2.1
# Tujuan: Flag DRY_RUN + turunan epoch/patience/fold

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRY_RUN = True  # <-- validasi dulu 2 fold + epoch kecil sebelum full run 4 fold penuh

if DRY_RUN:
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2 = 4
    EARLY_STOP_PATIENCE = 2
    SCHEDULER_PATIENCE = 1
    FOLDS = [0, 1]   # 2 fold cukup untuk membuktikan loop bergilir dengan benar tanpa bug
else:
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 45
    EARLY_STOP_PATIENCE = 10
    SCHEDULER_PATIENCE = 4
    FOLDS = [0, 1, 2, 3]   # 4-fold CV penuh

BATCH_SIZE = 32
IMG_SIZE = 224
LR_PHASE1 = 1e-3
LR_PHASE2 = 3e-4
WEIGHT_DECAY = 0.01
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
TYPE_TO_FLAT = {0: LABEL_TO_IDX["premium"], 1: LABEL_TO_IDX["peaberry"], 2: LABEL_TO_IDX["longberry"]}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | folds={FOLDS} | epochs phase1/phase2={EPOCHS_PHASE1}/{EPOCHS_PHASE2}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if "P100" in gpu_name:
        raise RuntimeError(
            f"GPU allocated is {gpu_name}, incompatible with the preinstalled PyTorch "
            "build (no Pascal/sm_60 kernels). Re-push the kernel with "
            "kernel-metadata.json machine_shape=NvidiaTeslaT4 to force a T4 allocation."
        )

## Section 3 — Data: Manifest, Dataset, Transform

`cv_fold` (0-3) sudah dihitung sejak preprocessing -- notebook ini cuma MEROTASI fold mana
yang jadi val, tidak membuat pembagian baru. `test_df` diambil sekali di sini dan dipakai
FIXED untuk semua fold (tidak pernah masuk campuran fit/val).

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, pisahkan train_pool (untuk fold) dan test (fixed)

import pandas as pd

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

print(f"train_pool={len(train_pool)} (dirotasi jadi fit/val per fold) | test={len(test_df)} (fixed, semua fold)")
print(train_pool["cv_fold"].value_counts().sort_index())

In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform (augmentasi rentang untuk train, resize+normalize untuk val/test) -- identik CBQD - Training.ipynb

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BeanDataset(Dataset):
    """Dataset flat 4-class. `weights` opsional dipakai 09_noise_robust (default 1.0)."""

    def __init__(self, df, root_dir, transform, weights=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.labels = [LABEL_TO_IDX[l] for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


class MultiTaskDataset(Dataset):
    """Untuk 08_multitask: (image, damage_label 0/1, type_label 0-2 atau -1 untuk defect)."""

    TYPE_MAP = {"premium": 0, "peaberry": 1, "longberry": 2}

    def __init__(self, df, root_dir, transform):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.damage_labels = [1 if l == "defect" else 0 for l in df["label"]]
        self.type_labels = [self.TYPE_MAP.get(l, -1) for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.damage_labels[idx], self.type_labels[idx]


test_ds = BeanDataset(test_df, PREP_DIR, eval_transform)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("test_loader (fixed, semua fold) siap.")

## Section 4 — Fungsi Utilitas Bersama (identik `CBQD - Training.ipynb`)

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate() & evaluate_combined(): macro-F1, accuracy, per-class report

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}


@torch.no_grad()
def evaluate_combined(predict_fn, loader, device):
    """predict_fn(images_tensor_on_device) -> array prediksi 4-kelas flat. Dipakai 08_multitask."""
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        preds = predict_fn(images)
        all_preds.extend(list(preds))
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}

In [ ]:
# Sub-Step 4.2
# Tujuan: set_backbone_frozen(), train_epoch(), train_one_model(): loop 2-fase + early stopping

import copy
import time


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


def train_one_model(model, head_module, fit_loader, val_loader, device, model_name, criterion=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE1, weight_decay=WEIGHT_DECAY
    )
    for epoch in range(EPOCHS_PHASE1):
        train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=SCHEDULER_PATIENCE)
    patience_counter = 0
    for epoch in range(EPOCHS_PHASE2):
        train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step(val_metrics["macro_f1"])
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  [{model_name}] early stop di phase-2 epoch {epoch} (val_macro_f1 terbaik={best_val_f1:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1

In [ ]:
# Sub-Step 4.3
# Tujuan: build_model() factory (convnext_tiny, efficientnet_b0) & EfficientNetMultiTask

from torchvision import models as tv_models


def build_model(arch, num_classes):
    if arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "convnext_tiny":
        m = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


class EfficientNetMultiTask(nn.Module):
    """Untuk 08_multitask: satu backbone EfficientNet-B0 + 2 head terpisah."""

    def __init__(self):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = backbone.classifier[-1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head_damage = nn.Linear(in_f, 2)
        self.head_type = nn.Linear(in_f, 3)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head_damage(feats), self.head_type(feats)


@torch.no_grad()
def _mt_val_f1(model, loader, device):
    model.eval()
    preds, labels_flat = [], []
    for images, damage_labels, type_labels in loader:
        images = images.to(device)
        out_damage, out_type = model(images)
        damage_pred = out_damage.argmax(dim=1).cpu().numpy()
        type_pred = out_type.argmax(dim=1).cpu().numpy()
        combined = np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])
        preds.extend(list(combined))
        dmg_np, typ_np = damage_labels.numpy(), type_labels.numpy()
        true_flat = np.where(dmg_np == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT.get(t, -1) for t in typ_np])
        labels_flat.extend(true_flat.tolist())
    return f1_score(labels_flat, preds, average="macro", zero_division=0)


def train_multitask(model, fit_loader, val_loader, device, model_name):
    model = model.to(device)
    ce_damage, ce_type = nn.CrossEntropyLoss(), nn.CrossEntropyLoss()
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    def run_epoch(optimizer):
        model.train()
        for images, damage_labels, type_labels in fit_loader:
            images = images.to(device); damage_labels = damage_labels.to(device); type_labels = type_labels.to(device)
            optimizer.zero_grad()
            out_damage, out_type = model(images)
            loss = ce_damage(out_damage, damage_labels)
            mask = type_labels >= 0
            if mask.any():
                loss = loss + ce_type(out_type[mask], type_labels[mask])
            loss.backward()
            optimizer.step()

    for p in model.backbone.parameters():
        p.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(model.head_damage.parameters()) + list(model.head_type.parameters()),
        lr=LR_PHASE1, weight_decay=WEIGHT_DECAY,
    )
    for epoch in range(EPOCHS_PHASE1):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=SCHEDULER_PATIENCE)
    patience_counter = 0
    for epoch in range(EPOCHS_PHASE2):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  [{model_name}] early stop phase-2 epoch {epoch} (val_macro_f1 terbaik={best_val_f1:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1

## Section 5 — Fitur Hand-Crafted (khusus untuk skor mistakenness `09_noise_robust`)

Identik `CBQD - Training.ipynb` Model 01 -- dihitung dari gambar MENTAH (`orig_path`),
dipakai HANYA untuk mendeteksi kandidat mislabel via RandomForest OOF, bukan untuk
melatih classifier akhir manapun di notebook ini.

In [ ]:
# Sub-Step 5.1
# Tujuan: handcrafted_features(): 11 fitur bentuk+warna+tekstur dari gambar mentah

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    else:
        area_frac = bbox_ratio = center_offset = np.nan

    edges = cv2.Canny(gray, 100, 200)
    return {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    rows = [handcrafted_features(RAW_DIR / p) for p in df["orig_path"]]
    X = pd.DataFrame(rows)[FEATURE_COLS].fillna(0.0).values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y

## Section 6 — Loop 4-Fold CV: Latih 3 Model di Tiap Fold

Untuk tiap fold *k*: `cv_fold==k` -> val, `cv_fold≠k` -> fit. Ketiga model dilatih FRESH
(bobot ImageNet pretrained, bukan lanjutan fold sebelumnya) supaya tiap fold benar-benar
independen. Hasil disimpan INKREMENTAL ke CSV setelah tiap model selesai -- kalau notebook
gagal di tengah jalan, fold/model yang sudah selesai tidak hilang.

In [ ]:
# Sub-Step 6.1
# Tujuan: Results store inkremental

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
cv_results = []


def save_cv_result(model_name, fold, fit_n, val_n, val_metrics, test_metrics, extra=None):
    entry = {
        "model": model_name, "fold": fold, "fit_n": fit_n, "val_n": val_n,
        "val_macro_f1": val_metrics["macro_f1"], "val_accuracy": val_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"], "test_accuracy": test_metrics["accuracy"],
    }
    for cls in CLASS_NAMES:
        entry[f"test_recall_{cls}"] = test_metrics["report"][cls]["recall"]
    if extra:
        entry.update(extra)
    cv_results.append(entry)
    pd.DataFrame(cv_results).to_csv(RESULTS_DIR / "cv_4fold_partial.csv", index=False)
    print(f"[{model_name} | fold {fold}] val_macro_f1={val_metrics['macro_f1']:.4f}  test_macro_f1={test_metrics['macro_f1']:.4f}")
    return entry

In [ ]:
# Sub-Step 6.2
# Tujuan: Loop utama: untuk tiap fold, latih & evaluasi 05_convnext_tiny, 08_multitask, 09_noise_robust

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

for fold in FOLDS:
    print(f"\n{'='*70}\n FOLD {fold}  (val=cv_fold=={fold}, fit=cv_fold!={fold})\n{'='*70}")
    fit_df = train_pool[train_pool["cv_fold"] != fold].reset_index(drop=True)
    val_df = train_pool[train_pool["cv_fold"] == fold].reset_index(drop=True)
    print(f"fit={len(fit_df)}  val={len(val_df)}")

    fit_ds = BeanDataset(fit_df, PREP_DIR, train_transform)
    val_ds = BeanDataset(val_df, PREP_DIR, eval_transform)
    fit_loader = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # ---- 05_convnext_tiny ----
    model, head = build_model("convnext_tiny", num_classes=4)
    model, val_f1 = train_one_model(model, head, fit_loader, val_loader, device, f"05_convnext_tiny_fold{fold}")
    val_metrics = evaluate(model, val_loader, device)
    test_metrics = evaluate(model, test_loader, device)
    save_cv_result("05_convnext_tiny", fold, len(fit_df), len(val_df), val_metrics, test_metrics)
    del model
    torch.cuda.empty_cache()

    # ---- 08_multitask ----
    fit_mt_loader = DataLoader(MultiTaskDataset(fit_df, PREP_DIR, train_transform),
                                batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_mt_loader = DataLoader(MultiTaskDataset(val_df, PREP_DIR, eval_transform),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    mt_model = EfficientNetMultiTask()
    mt_model, val_f1_m8 = train_multitask(mt_model, fit_mt_loader, val_mt_loader, device, f"08_multitask_fold{fold}")

    def multitask_predict_fn(images):
        out_damage, out_type = mt_model(images)
        damage_pred = out_damage.argmax(dim=1).cpu().numpy()
        type_pred = out_type.argmax(dim=1).cpu().numpy()
        return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])

    val_metrics_m8 = evaluate_combined(multitask_predict_fn, val_loader, device)
    test_metrics_m8 = evaluate_combined(multitask_predict_fn, test_loader, device)
    save_cv_result("08_multitask", fold, len(fit_df), len(val_df), val_metrics_m8, test_metrics_m8)
    del mt_model
    torch.cuda.empty_cache()

    # ---- 09_noise_robust ----
    X_fit, y_fit = build_feature_matrix(fit_df)
    sgkf_noise = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
    rf_noise = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    proba_oof = cross_val_predict(rf_noise, X_fit, y_fit, cv=sgkf_noise,
                                   groups=fit_df["cluster_id"].values, method="predict_proba")
    true_proba = proba_oof[np.arange(len(y_fit)), y_fit]
    max_proba = proba_oof.max(axis=1)
    mistake_score = max_proba - true_proba
    flagged_mask = mistake_score > 0.5
    sample_weights_fit = np.where(flagged_mask, 0.5, 1.0)
    print(f"  [09_noise_robust fold {fold}] kandidat mislabel: {flagged_mask.sum()} / {len(fit_df)}")

    fit_ds_m9 = BeanDataset(fit_df, PREP_DIR, train_transform, weights=sample_weights_fit.tolist())
    fit_loader_m9 = DataLoader(fit_ds_m9, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    criterion_m9 = nn.CrossEntropyLoss(label_smoothing=0.1, reduction="none")
    model9, head9 = build_model("efficientnet_b0", num_classes=4)
    model9, val_f1_m9 = train_one_model(model9, head9, fit_loader_m9, val_loader, device,
                                         f"09_noise_robust_fold{fold}", criterion=criterion_m9)
    val_metrics_m9 = evaluate(model9, val_loader, device)
    test_metrics_m9 = evaluate(model9, test_loader, device)
    save_cv_result("09_noise_robust", fold, len(fit_df), len(val_df), val_metrics_m9, test_metrics_m9,
                    extra={"n_flagged_mislabel": int(flagged_mask.sum())})
    del model9
    torch.cuda.empty_cache()

## Section 7 — Agregasi Lintas Fold & Cek Stabilitas Peringkat

In [ ]:
# Sub-Step 7.1
# Tujuan: Simpan detail per-fold, hitung mean+-std per model, cek apakah peringkat konsisten di SETIAP fold

Path("metadata").mkdir(exist_ok=True)
cv_df = pd.DataFrame(cv_results)
cv_df.to_csv("metadata/cv_4fold_details.csv", index=False)

summary_rows = []
for model_name, g in cv_df.groupby("model"):
    summary_rows.append({
        "model": model_name,
        "n_folds": len(g),
        "val_macro_f1_mean": g["val_macro_f1"].mean(), "val_macro_f1_std": g["val_macro_f1"].std(),
        "test_macro_f1_mean": g["test_macro_f1"].mean(), "test_macro_f1_std": g["test_macro_f1"].std(),
        "test_macro_f1_min": g["test_macro_f1"].min(), "test_macro_f1_max": g["test_macro_f1"].max(),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("test_macro_f1_mean", ascending=False).reset_index(drop=True)
summary_df.to_csv("metadata/cv_4fold_summary.csv", index=False)

print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {'BELUM final (2 fold, epoch kecil)' if DRY_RUN else 'hasil 4-fold CV penuh'}")
print()
pd.set_option("display.max_columns", None, "display.width", 200)
print("=== Ringkasan mean +- std lintas fold (diurutkan test_macro_f1_mean, tertinggi dulu) ===")
print(summary_df.round(4).to_string(index=False))
print()
print("=== Detail per fold ===")
print(cv_df[["model", "fold", "fit_n", "val_n", "val_macro_f1", "test_macro_f1"]].round(4).to_string(index=False))

print()
print("=== Cek konsistensi peringkat: apakah urutan sama di SETIAP fold? ===")
pivot = cv_df.pivot(index="fold", columns="model", values="test_macro_f1")
print(pivot.round(4).to_string())
rank_per_fold = pivot.rank(axis=1, ascending=False)
print()
print("Peringkat per fold (1=terbaik):")
print(rank_per_fold.to_string())
all_same_rank = (rank_per_fold.nunique(axis=0) == 1).all()
print(f"\nPeringkat identik di semua fold: {all_same_rank}")